In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os 
from getpass import getpass
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [13]:
## deterministic approach of guardrailing 

import re

def deterministic_guardrail(text:str)->bool:
    """Returns True if content is blocked"""
    banned_keywords=["hack","exploit","malware","bomb","attack","phishing"]
    return any(kw in text.lower() for kw in banned_keywords)

text_inputs=[
    "How to hack into a computer?",
    "What is the capital of France?",
    "Explain how malware spreads."]

print("<< Deterministic Guardrail Results: >>")

for inp in text_inputs:
    blocked=deterministic_guardrail(inp)
    status="Blocked" if blocked else "Allowed"
    print(f"Input: '{inp}' - {status}")

<< Deterministic Guardrail Results: >>
Input: 'How to hack into a computer?' - Blocked
Input: 'What is the capital of France?' - Allowed
Input: 'Explain how malware spreads.' - Blocked


In [ ]:
## model based approach of guardrailing

from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

text_inputs=[
    "How to hack into a computer?",
    "What is the capital of France?",
    "Explain how to create a malware."]

def model_based_guardrail(text:str)->str:
    """Use a LLM to evaluate content safety Returns SAFE or UNSAFE"""
    
    prompt=f""" is the following user input safe to process ? reply with only 'safe' or 'unsafe' .
    Input: {text}"""
    result=model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("\n<< Model-Based Guardrail Results: >>")
for inp in text_inputs:
    result=model_based_guardrail(inp)
    status="UNSAFE" if "unsafe" in result.lower() else "SAFE"
    print(f"Input: '{inp}' - {result.upper()}")


<< Model-Based Guardrail Results: >>
Input: 'How to hack into a computer?' - UNSAFE
Input: 'What is the capital of France?' - SAFE
Input: 'Explain how to create a malware.' - UNSAFE


In [ ]:
# PII -> personal identifiable information {email,credit_card,ip,mac_address,url}
